# 02 — Byte-Level BPE Tokenizer From Scratch

This notebook builds a **Byte-Level BPE tokenizer from scratch**.

The goal is not to use an existing tokenizer library, but to understand:

- Why language models need tokenizers
- Why word-level tokenization is limited
- What subword tokenization solves
- How Byte Pair Encoding (BPE) works
- Why byte-level BPE starts from 256 possible byte values
- How BPE learns merge rules from a corpus
- How text is converted into token IDs
- How token IDs are decoded back into text
- How vocabulary size affects tokenization
- How tokenization changes the number of tokens seen during GPT training

## Learning Pipeline

```text
Raw Text
   ↓
UTF-8 Encoding
   ↓
Bytes
   ↓
Initial Byte Vocabulary
   ↓
Count Adjacent Token Pairs
   ↓
Merge Most Frequent Pair
   ↓
Repeat
   ↓
Learned Vocabulary + Merge Rules
   ↓
Encode Text → Token IDs
   ↓
Decode Token IDs → Text

In [43]:
words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]


def count_pairs(words):
    pair_counts = {}

    for word in words:
        for pair in zip(word, word[1:]):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

    return pair_counts


pair_counts = count_pairs(words)

print(pair_counts)

{('p', 'l'): 8, ('l', 'a'): 8, ('a', 'y'): 8, ('y', 'e'): 3, ('e', 'd'): 2, ('y', 'i'): 2, ('i', 'n'): 2, ('n', 'g'): 2, ('e', 'r'): 1, ('y', 's'): 1, ('r', 'e'): 3, ('e', 'p'): 3}


In [46]:
def merge_pair(tokens, pair):
    merged = []
    i = 0

    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1

    return merged

# tokens = list("played")

pair = ('p', 'l')

for word in words:
    result = merge_pair(list(word), list(pair_counts)[0])

    print(f'{list(word)} --> {result}')

['p', 'l', 'a', 'y'] --> ['pl', 'a', 'y']
['p', 'l', 'a', 'y', 'e', 'd'] --> ['pl', 'a', 'y', 'e', 'd']
['p', 'l', 'a', 'y', 'i', 'n', 'g'] --> ['pl', 'a', 'y', 'i', 'n', 'g']
['p', 'l', 'a', 'y', 'e', 'r'] --> ['pl', 'a', 'y', 'e', 'r']
['p', 'l', 'a', 'y', 's'] --> ['pl', 'a', 'y', 's']
['r', 'e', 'p', 'l', 'a', 'y'] --> ['r', 'e', 'pl', 'a', 'y']
['r', 'e', 'p', 'l', 'a', 'y', 'e', 'd'] --> ['r', 'e', 'pl', 'a', 'y', 'e', 'd']
['r', 'e', 'p', 'l', 'a', 'y', 'i', 'n', 'g'] --> ['r', 'e', 'pl', 'a', 'y', 'i', 'n', 'g']


In [41]:
list({'a':0, 'b':1})


['a', 'b']